# Feature extraction — TF-IDF (Phase 3)

Cleaned tweets are still **strings**; models need **numbers**. **TF-IDF** turns each document into a vector: each dimension is a **term** from the vocabulary, and the value encodes how **distinctive** that term is for that document relative to the corpus.

- **TF** (term frequency): how often the term appears in this document.
- **IDF** (inverse document frequency): down-weights terms that appear in *many* documents (e.g. “the”, “this”) and up-weights **rarer**, more informative terms.

This notebook reads **`data/processed_dataset.csv`** from preprocessing (`text_clean` is the same role as `cleaned_text` in generic tutorials).

## How TF-IDF scoring works (preview)

The cell below uses a **tiny toy corpus** so you can see the TF-IDF matrix before training on real data. Words that show up in almost every line get **low IDF**; distinctive words get **higher** weights.

In [2]:
%pip install -q pandas scikit-learn scipy joblib ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from __future__ import annotations

from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_CSV = DATA_DIR / "processed_dataset.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.20

VECTORIZER_PATH = MODELS_DIR / "tfidf_vectorizer.pkl"

In [4]:
toy_docs = [
    "the movie was good but the plot was slow",
    "the the the boring movie",
    "excellent plot excellent acting",
]
toy_vec = TfidfVectorizer(sublinear_tf=True)
Xtoy = toy_vec.fit_transform(toy_docs)
names = toy_vec.get_feature_names_out()
toy_dense = Xtoy.toarray()
display(pd.DataFrame(toy_dense, columns=names, index=[f"doc{i}" for i in range(len(toy_docs))]))
idf_series = pd.Series(toy_vec.idf_, index=names).sort_values()
print("Lowest IDF (appear in many docs — less informative):", idf_series.head(6).to_dict())
print("Highest IDF (rarer — more distinctive):", idf_series.tail(6).to_dict())

,acting,boring,but,excellent,good,movie,plot,slow,the,was
doc0,0.000000,0.00000,0.339389,0.000000,0.339389,0.258114,0.258114,0.339389,0.437026,0.574636
doc1,0.000000,0.49232,0.000000,0.000000,0.000000,0.374422,0.000000,0.000000,0.785767,0.000000
doc2,0.474304,0.00000,0.000000,0.803067,0.000000,0.000000,0.360721,0.000000,0.000000,0.000000


Lowest IDF (appear in many docs — less informative): {'plot': 1.2876820724517808, 'movie': 1.2876820724517808, 'the': 1.2876820724517808, 'acting': 1.6931471805599454, 'excellent': 1.6931471805599454, 'but': 1.6931471805599454}
Highest IDF (rarer — more distinctive): {'excellent': 1.6931471805599454, 'but': 1.6931471805599454, 'good': 1.6931471805599454, 'boring': 1.6931471805599454, 'slow': 1.6931471805599454, 'was': 1.6931471805599454}


In [5]:
df = pd.read_csv(PROCESSED_CSV)
df = df.dropna(subset=["text_clean", "target"])
df["text_clean"] = df["text_clean"].astype(str).str.strip()
df = df[df["text_clean"].str.len() > 0]

X = df["text_clean"]
# Multiclass labels: 0 = negative, 1 = neutral, 2 = positive (see sentiment_analysis.ipynb)
y = df["target"].astype(int)
if not set(y.unique()).issubset({0, 1, 2}):
    raise ValueError("Expected target in {0,1,2}; regenerate data with sentiment_analysis.ipynb")

print("Rows:", len(df))
print(y.value_counts().rename(index={0: "neg(0)", 1: "pos(1)"}))

Rows: 4500
target
pos(1)    1500
neg(0)    1500
2         1500
Name: count, dtype: int64


In [6]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train_raw.shape[0], "  Test:", X_test_raw.shape[0])

Train: 3600   Test: 900


In [7]:
# Full pipeline: unigrams + bigrams, sublinear TF, frequency filters.
# On very small corpora, min_df=3 can over-prune — relax if vocabulary collapses.
_min_df = 3 if X_train_raw.shape[0] >= 500 else 1

vectorizer = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),
    min_df=_min_df,
    max_df=0.90,
    sublinear_tf=True,
)

# Fit on TRAIN only — never fit on test (avoids leakage)
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

In [8]:
vocab_size = len(vectorizer.vocabulary_)
density = X_train.nnz / (X_train.shape[0] * X_train.shape[1])

print(f"Vocabulary size : {vocab_size}")
print(f"X_train shape   : {X_train.shape}")
print(f"X_test shape    : {X_test.shape}")
print(f"Matrix density  : {density:.4%}")

print("\nReady for Phase 4: X_train, X_test, y_train, y_test + saved vectorizer.")

Vocabulary size : 69
X_train shape   : (3600, 69)
X_test shape    : (900, 69)
Matrix density  : 4.4758%

Ready for Phase 4: X_train, X_test, y_train, y_test + saved vectorizer.


### Explore a training document (optional widget)

Pick a **document index** and a **word** that appears in the vocabulary. The table shows that term’s **IDF** (corpus-wide) and its **TF-IDF weight** in that document (after vectorizer normalization). Rare words tend to have higher IDF; stopword-like terms are usually filtered by `min_df` / `max_df`.

In [ ]:
fnames = vectorizer.get_feature_names_out()
vocab = vectorizer.vocabulary_
idf = vectorizer.idf_


def word_breakdown(doc_idx: int, word: str):
    word = word.strip().lower()
    if word not in vocab:
        print(f"'{word}' not in vocabulary (try another token or ngram).")
        return
    j = vocab[word]
    row = X_train.getrow(doc_idx)
    tfidf_w = float(row[0, j]) if j in row.indices else 0.0
    display(
        pd.DataFrame(
            [{"word": word, "idf": float(idf[j]), "tfidf_in_doc": tfidf_w}]
        )
    )


def top_terms_in_doc(doc_idx: int, k: int = 15):
    row = X_train.getrow(doc_idx)
    pairs = [(fnames[i], float(row[0, i])) for i in row.indices]
    pairs.sort(key=lambda x: -x[1])
    display(pd.DataFrame(pairs[:k], columns=["term", "tf-idf weight"]))


sample_doc = 0
print("Top TF-IDF terms in training document", sample_doc, "(first line of train):")
print(repr(X_train_raw.iloc[sample_doc]))
top_terms_in_doc(sample_doc)

try:
    import ipywidgets as W

    word_options = list(fnames[: min(200, len(fnames))])

    W.interact(
        lambda doc_idx, word: word_breakdown(int(doc_idx), word),
        doc_idx=W.IntSlider(0, 0, X_train.shape[0] - 1, 1, description="doc_idx"),
        word=W.Dropdown(options=word_options, description="word"),
    )
except ImportError:
    print("(Install ipywidgets for interactive sliders: already in pip cell above.)")
    if len(fnames):
        word_breakdown(0, str(fnames[0]))

Top TF-IDF terms in training document 0 (first line of train):
'link attach reference'


,term,tf-idf weight
0,link,0.447214
1,attach,0.447214
2,reference,0.447214
3,link attach,0.447214
4,attach reference,0.447214


In [ ]:
joblib.dump(vectorizer, VECTORIZER_PATH)
print("Vectorizer saved:", VECTORIZER_PATH)